In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 36.4 MB/s eta 0:00:00


In [2]:
import numpy as np
import os
try:
    import gensim.downloader as api
    from gensim.models import KeyedVectors
    GENSIM_AVAILABLE = True
except ImportError:
    GENSIM_AVAILABLE = False
    print("Warning: gensim not installed")

In [3]:
def load_embeddings_from_text(filepath, limit=None):
    embeddings = {}
    print(f"Loading embeddings from {filepath}...")

    with open(filepath, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if limit and i >= limit:
                break

            values = line.strip().split()
            if len(values) < 2:
                continue

            word = values[0]
            try:
                vector = np.array(values[1:], dtype='float32')
                embeddings[word] = vector
            except:
                continue

            if (i + 1) % 100000 == 0:
                print(f"  Loaded {i + 1} words...")

    print(f"Loaded {len(embeddings)} words")
    return embeddings

In [4]:
def load_word2vec_300(filepath=None):
    print("\n[Loading Word2Vec 300]")

    if filepath and os.path.exists(filepath):
        if filepath.endswith('.bin'):
            if GENSIM_AVAILABLE:
                model = KeyedVectors.load_word2vec_format(filepath, binary=True)
                print(f"Vocabulary size: {len(model)}")
                return model
        else:
            return load_embeddings_from_text(filepath)

    if GENSIM_AVAILABLE:
        try:
            model = api.load('word2vec-google-news-300')
            print(f"Vocabulary size: {len(model)}")
            return model
        except:
            print("Download failed. Provide filepath.")
            return None

    print("gensim not available. Provide filepath.")
    return None

In [5]:
def load_glove_300(filepath=None):
    print("\n[Loading GloVe 300]")

    if filepath and os.path.exists(filepath):
        return load_embeddings_from_text(filepath)

    if GENSIM_AVAILABLE:
        try:
            model = api.load('glove-wiki-gigaword-300')
            print(f"Vocabulary size: {len(model)}")
            return model
        except:
            print("Download failed. Provide filepath.")
            return None

    print("gensim not available. Provide filepath.")
    return None


In [6]:
def load_glove_100(filepath=None):
    print("\n[Loading GloVe 100]")

    if filepath and os.path.exists(filepath):
        return load_embeddings_from_text(filepath)

    if GENSIM_AVAILABLE:
        try:
            model = api.load('glove-wiki-gigaword-100')
            print(f"Vocabulary size: {len(model)}")
            return model
        except:
            print("Download failed. Provide filepath.")
            return None

    print("gensim not available. Provide filepath.")
    return None

In [7]:
def load_fasttext_300(filepath=None):
    print("\n[Loading fastText 300]")

    if filepath and os.path.exists(filepath):
        if filepath.endswith('.bin'):
            if GENSIM_AVAILABLE:
                from gensim.models.fasttext import load_facebook_model
                model = load_facebook_model(filepath)
                print(f"Vocabulary size: {len(model.wv)}")
                return model.wv
        else:
            return load_embeddings_from_text(filepath)

    if GENSIM_AVAILABLE:
        try:
            model = api.load('fasttext-wiki-news-subwords-300')
            print(f"Vocabulary size: {len(model)}")
            return model
        except:
            print("Download failed. Provide filepath.")
            return None

    print("gensim not available. Provide filepath.")
    return None

In [8]:
def load_glove_50(filepath=None):
    print("\n[Loading GloVe 50]")

    if filepath and os.path.exists(filepath):
        return load_embeddings_from_text(filepath)

    if GENSIM_AVAILABLE:
        try:
            model = api.load('glove-wiki-gigaword-50')
            print(f"Vocabulary size: {len(model)}")
            return model
        except:
            print("Download failed. Provide filepath.")
            return None

    print("gensim not available. Provide filepath.")
    return None

In [9]:
def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)

    if norm_vec1 == 0 or norm_vec2 == 0:
        return 0.0

    return dot_product / (norm_vec1 * norm_vec2)

In [10]:
def solve_analogy(word_a, word_b, word_c, embeddings, top_n=5, exclude_input_words=True):
    try:
        if isinstance(embeddings, dict):
            if word_a not in embeddings or word_b not in embeddings or word_c not in embeddings:
                return None

            vec_a = embeddings[word_a]
            vec_b = embeddings[word_b]
            vec_c = embeddings[word_c]
            vocab = embeddings.keys()
        else:
            if word_a not in embeddings or word_b not in embeddings or word_c not in embeddings:
                return None

            vec_a = embeddings[word_a]
            vec_b = embeddings[word_b]
            vec_c = embeddings[word_c]
            vocab = embeddings.index_to_key

        target_vector = vec_b - vec_a + vec_c

        similarities = []

        for word in vocab:
            if exclude_input_words and word in [word_a, word_b, word_c]:
                continue

            if isinstance(embeddings, dict):
                word_vec = embeddings[word]
            else:
                word_vec = embeddings[word]

            sim = cosine_similarity(target_vector, word_vec)
            similarities.append((word, sim))

        similarities.sort(key=lambda x: x[1], reverse=True)

        return similarities[:top_n]

    except Exception as e:
        return None

In [70]:
SYNTACTIC_ANALOGIES = [
    ("run", "running", "swim", "swimming"),
    ("govern", "government", "achieve", "achievement"),
    ("mouse", "mice", "goose", "geese"),
    ("happy", "happier", "heavy", "heavier"),
    ("quick", "quickly", "slow", "slowly"),
    ("write", "writer", "teach", "teacher"),
    ("develop", "development", "argue", "argument"),
    ("create", "creation", "relate", "relation"),
    ("nation", "national", "culture", "cultural"),
    ("music", "musical", "history", "historical"),
]

SEMANTIC_ANALOGIES = [
    ("paris", "france", "rome", "italy"),
    ("apple", "fruit", "spinach", "vegetables"),
    ("car", "driver", "airplane", "pilots"),
    ("japan", "yen", "india", "rupee"),
    ("camera", "photo", "mic", "sound"),
    ("doctor", "hospital", "teacher", "school"),
    ("light", "dark", "day", "night"),
    ("seed", "plant", "caterpillar", "butterfly"),
    ("tokyo", "japan", "ottawa", "canada"),
    ("dawn", "sunrise", "dusk", "sunset"),
]

In [12]:
w2v_300 = load_word2vec_300()
glove_300 = load_glove_300()
glove_100 = load_glove_100()
fasttext_300 = load_fasttext_300()
glove_50 = load_glove_50()


[Loading Word2Vec 300]
[==================================================] 100.0% 1662.8/1662.8MB downloaded
Vocabulary size: 3000000

[Loading GloVe 300]
[==================================================] 100.0% 376.1/376.1MB downloaded
Vocabulary size: 400000

[Loading GloVe 100]
[==================================================] 100.0% 128.1/128.1MB downloaded
Vocabulary size: 400000

[Loading fastText 300]
[==================================================] 100.0% 958.5/958.4MB downloaded
Vocabulary size: 999999

[Loading GloVe 50]
[==================================================] 100.0% 66.0/66.0MB downloaded
Vocabulary size: 400000


In [71]:
print("PART 1: Word2Vec 300 vs GloVe 300 vs fastText 300")

print("\nTesting Semantic Analogies...")
sem_results_w2v = []
sem_results_glove300 = []
sem_results_fasttext300 = []

for word_a, word_b, word_c, expected in SEMANTIC_ANALOGIES:
    result = solve_analogy(word_a, word_b, word_c, w2v_300, top_n=5)
    pred_w2v = result[0][0] if result else "N/A"

    result = solve_analogy(word_a, word_b, word_c, glove_300, top_n=5)
    pred_glove = result[0][0] if result else "N/A"

    result = solve_analogy(word_a, word_b, word_c, fasttext_300, top_n=5)
    pred_fasttext = result[0][0] if result else "N/A"

    sem_results_w2v.append(pred_w2v)
    sem_results_glove300.append(pred_glove)
    sem_results_fasttext300.append(pred_fasttext)

PART 1: Word2Vec 300 vs GloVe 300 vs fastText 300

Testing Semantic Analogies...


In [72]:
print("Syntactic : Word2Vec 300 vs GloVe 300 vs fastText 300")
syn_results_w2v = []
syn_results_glove300 = []
syn_results_fasttext300 = []

for word_a, word_b, word_c, expected in SYNTACTIC_ANALOGIES:
    result = solve_analogy(word_a, word_b, word_c, w2v_300, top_n=5)
    pred_w2v = result[0][0] if result else "N/A"

    result = solve_analogy(word_a, word_b, word_c, glove_300, top_n=5)
    pred_glove = result[0][0] if result else "N/A"

    result = solve_analogy(word_a, word_b, word_c, fasttext_300, top_n=5)
    pred_fasttext = result[0][0] if result else "N/A"

    syn_results_w2v.append(pred_w2v)
    syn_results_glove300.append(pred_glove)
    syn_results_fasttext300.append(pred_fasttext)

Syntactic : Word2Vec 300 vs GloVe 300 vs fastText 300


In [73]:
print("Semantic : Word2Vec 300 vs GloVe 300 vs fastText 300")

print(f"{'Test #':<8} {'Test':<40} {'Expected':<15} {'GloVe 300':<15} {'Word2Vec 300':<15} {'fastText 300':<15}")
print("-"*110)

for i, (a, b, c, expected) in enumerate(SEMANTIC_ANALOGIES, 1):
    test_str = f"{a}:{b}::{c}:?"
    glove_result = sem_results_glove300[i-1]
    w2v_result = sem_results_w2v[i-1]
    fasttext_result = sem_results_fasttext300[i-1]
    print(f"{i:<8} {test_str:<40} {expected:<15} {glove_result:<15} {w2v_result:<15} {fasttext_result:<15}")

Semantic : Word2Vec 300 vs GloVe 300 vs fastText 300
Test #   Test                                     Expected        GloVe 300       Word2Vec 300    fastText 300   
--------------------------------------------------------------------------------------------------------------
1        paris:france::rome:?                     italy           italy           italy           germany        
2        apple:fruit::spinach:?                   vegetables      vegetables      fresh_spinach   vegetables     
3        car:driver::airplane:?                   pilots          plane           plane           pilots         
4        japan:yen::india:?                       rupee           rupees          rupee           rupee          
5        camera:photo::mic:?                      sound           someșul         mike            mike           
6        doctor:hospital::teacher:?               school          school          elementary      school         
7        light:dark::day:?            

In [74]:

print("Syntactic - Word2Vec 300 vs GloVe 300 vs fastText 300")

print(f"{'Test #':<8} {'Test':<40} {'Expected':<15} {'GloVe 300':<15} {'Word2Vec 300':<15} {'fastText 300':<15}")
print("-"*110)

for i, (a, b, c, expected) in enumerate(SYNTACTIC_ANALOGIES, 1):
    test_str = f"{a}:{b}::{c}:?"
    glove_result = syn_results_glove300[i-1]
    w2v_result = syn_results_w2v[i-1]
    fasttext_result = syn_results_fasttext300[i-1]
    print(f"{i:<8} {test_str:<40} {expected:<15} {glove_result:<15} {w2v_result:<15} {fasttext_result:<15}")

Syntactic - Word2Vec 300 vs GloVe 300 vs fastText 300
Test #   Test                                     Expected        GloVe 300       Word2Vec 300    fastText 300   
--------------------------------------------------------------------------------------------------------------
1        run:running::swim:?                      swimming        swimming        swimming        swimming       
2        govern:government::achieve:?             achievement     achieved        achieving       achieves       
3        mouse:mice::goose:?                      geese           geese           geese           geese          
4        happy:happier::heavy:?                   heavier         heavier         heavier         heavier        
5        quick:quickly::slow:?                    slowly          slowly          slowly          slowly         
6        write:writer::teach:?                    teacher         teaches         taught          teacher        
7        develop:development::argue:?

In [75]:

sem_correct_glove300 = sum(1 for i, (_, _, _, exp) in enumerate(SEMANTIC_ANALOGIES)
                           if sem_results_glove300[i].lower() == exp.lower())
sem_correct_w2v = sum(1 for i, (_, _, _, exp) in enumerate(SEMANTIC_ANALOGIES)
                      if sem_results_w2v[i].lower() == exp.lower())
sem_correct_fasttext300 = sum(1 for i, (_, _, _, exp) in enumerate(SEMANTIC_ANALOGIES)
                              if sem_results_fasttext300[i].lower() == exp.lower())

sem_acc_glove300 = (sem_correct_glove300 / len(SEMANTIC_ANALOGIES)) * 100
sem_acc_w2v = (sem_correct_w2v / len(SEMANTIC_ANALOGIES)) * 100
sem_acc_fasttext300 = (sem_correct_fasttext300 / len(SEMANTIC_ANALOGIES)) * 100

syn_correct_glove300 = sum(1 for i, (_, _, _, exp) in enumerate(SYNTACTIC_ANALOGIES)
                           if syn_results_glove300[i].lower() == exp.lower())
syn_correct_w2v = sum(1 for i, (_, _, _, exp) in enumerate(SYNTACTIC_ANALOGIES)
                      if syn_results_w2v[i].lower() == exp.lower())
syn_correct_fasttext300 = sum(1 for i, (_, _, _, exp) in enumerate(SYNTACTIC_ANALOGIES)
                              if syn_results_fasttext300[i].lower() == exp.lower())

syn_acc_glove300 = (syn_correct_glove300 / len(SYNTACTIC_ANALOGIES)) * 100
syn_acc_w2v = (syn_correct_w2v / len(SYNTACTIC_ANALOGIES)) * 100
syn_acc_fasttext300 = (syn_correct_fasttext300 / len(SYNTACTIC_ANALOGIES)) * 100

print("ACCURACY SUMMARY")
print(f"{'Analogy Type':<20} {'GloVe 300':<20} {'Word2Vec 300':<20} {'fastText 300':<20}")
print("-"*80)
print(f"{'Semantic':<20} {sem_acc_glove300:<19.2f}% {sem_acc_w2v:<19.2f}% {sem_acc_fasttext300:<19.2f}%")
print(f"{'Syntactic':<20} {syn_acc_glove300:<19.2f}% {syn_acc_w2v:<19.2f}% {syn_acc_fasttext300:<19.2f}%")

ACCURACY SUMMARY
Analogy Type         GloVe 300            Word2Vec 300         fastText 300        
--------------------------------------------------------------------------------
Semantic             50.00              % 40.00              % 70.00              %
Syntactic            60.00              % 50.00              % 70.00              %


In [76]:
print(" GloVe 300 vs GloVe 100 vs GloVe 50")

print("\nSemantic Analogies")
sem_results_glove300 = []
sem_results_glove100 = []
sem_results_glove50 = []

for word_a, word_b, word_c, expected in SEMANTIC_ANALOGIES:
    result = solve_analogy(word_a, word_b, word_c, glove_300, top_n=5)
    pred_glove300 = result[0][0] if result else "N/A"

    result = solve_analogy(word_a, word_b, word_c, glove_100, top_n=5)
    pred_glove100 = result[0][0] if result else "N/A"

    result = solve_analogy(word_a, word_b, word_c, glove_50, top_n=5)
    pred_glove50 = result[0][0] if result else "N/A"

    sem_results_glove300.append(pred_glove300)
    sem_results_glove100.append(pred_glove100)
    sem_results_glove50.append(pred_glove50)


 GloVe 300 vs GloVe 100 vs GloVe 50

Semantic Analogies


In [77]:
print("\n Syntactic Analogies")
syn_results_glove300 = []
syn_results_glove100 = []
syn_results_glove50 = []

for word_a, word_b, word_c, expected in SYNTACTIC_ANALOGIES:
    result = solve_analogy(word_a, word_b, word_c, glove_300, top_n=5)
    pred_glove300 = result[0][0] if result else "N/A"

    result = solve_analogy(word_a, word_b, word_c, glove_100, top_n=5)
    pred_glove100 = result[0][0] if result else "N/A"

    result = solve_analogy(word_a, word_b, word_c, glove_50, top_n=5)
    pred_glove50 = result[0][0] if result else "N/A"

    syn_results_glove300.append(pred_glove300)
    syn_results_glove100.append(pred_glove100)
    syn_results_glove50.append(pred_glove50)



 Syntactic Analogies


In [78]:
print("Syntactic : GloVe 300 vs GloVe 100 vs GloVe 50")

print(f"{'Test #':<8} {'Test':<40} {'Expected':<15} {'GloVe 300':<15} {'GloVe 100':<15} {'GloVe 50':<15}")
print("-"*110)

for i, (a, b, c, expected) in enumerate(SYNTACTIC_ANALOGIES, 1):
    test_str = f"{a}:{b}::{c}:?"
    glove300_result = syn_results_glove300[i-1]
    glove100_result = syn_results_glove100[i-1]
    glove50_result = syn_results_glove50[i-1]
    print(f"{i:<8} {test_str:<40} {expected:<15} {glove300_result:<15} {glove100_result:<15} {glove50_result:<15}")

Syntactic : GloVe 300 vs GloVe 100 vs GloVe 50
Test #   Test                                     Expected        GloVe 300       GloVe 100       GloVe 50       
--------------------------------------------------------------------------------------------------------------
1        run:running::swim:?                      swimming        swimming        swimming        swimming       
2        govern:government::achieve:?             achievement     achieved        progress        support        
3        mouse:mice::goose:?                      geese           geese           cows            calves         
4        happy:happier::heavy:?                   heavier         heavier         heavier         heavier        
5        quick:quickly::slow:?                    slowly          slowly          rapidly         rapidly        
6        write:writer::teach:?                    teacher         teaches         teacher         sociologist    
7        develop:development::argue:?       

In [79]:

print("Semantic - GloVe 300 vs GloVe 100 vs GloVe 50")

print(f"{'Test #':<8} {'Test':<40} {'Expected':<15} {'GloVe 300':<15} {'GloVe 100':<15} {'GloVe 50':<15}")
print("-"*110)

for i, (a, b, c, expected) in enumerate(SEMANTIC_ANALOGIES, 1):
    test_str = f"{a}:{b}::{c}:?"
    glove300_result = sem_results_glove300[i-1]
    glove100_result = sem_results_glove100[i-1]
    glove50_result = sem_results_glove50[i-1]
    print(f"{i:<8} {test_str:<40} {expected:<15} {glove300_result:<15} {glove100_result:<15} {glove50_result:<15}")

Semantic - GloVe 300 vs GloVe 100 vs GloVe 50
Test #   Test                                     Expected        GloVe 300       GloVe 100       GloVe 50       
--------------------------------------------------------------------------------------------------------------
1        paris:france::rome:?                     italy           italy           italy           italy          
2        apple:fruit::spinach:?                   vegetables      vegetables      lettuce         lettuce        
3        car:driver::airplane:?                   pilots          plane           pilot           pilot          
4        japan:yen::india:?                       rupee           rupees          rupees          rupees         
5        camera:photo::mic:?                      sound           someșul         gazette         ajc            
6        doctor:hospital::teacher:?               school          school          school          school         
7        light:dark::day:?                   

In [80]:
sem_correct_glove300 = sum(1 for i, (_, _, _, exp) in enumerate(SEMANTIC_ANALOGIES)
                           if sem_results_glove300[i].lower() == exp.lower())
sem_correct_glove100 = sum(1 for i, (_, _, _, exp) in enumerate(SEMANTIC_ANALOGIES)
                           if sem_results_glove100[i].lower() == exp.lower())
sem_correct_glove50 = sum(1 for i, (_, _, _, exp) in enumerate(SEMANTIC_ANALOGIES)
                          if sem_results_glove50[i].lower() == exp.lower())

sem_acc_glove300 = (sem_correct_glove300 / len(SEMANTIC_ANALOGIES)) * 100
sem_acc_glove100 = (sem_correct_glove100 / len(SEMANTIC_ANALOGIES)) * 100
sem_acc_glove50 = (sem_correct_glove50 / len(SEMANTIC_ANALOGIES)) * 100

syn_correct_glove300 = sum(1 for i, (_, _, _, exp) in enumerate(SYNTACTIC_ANALOGIES)
                           if syn_results_glove300[i].lower() == exp.lower())
syn_correct_glove100 = sum(1 for i, (_, _, _, exp) in enumerate(SYNTACTIC_ANALOGIES)
                           if syn_results_glove100[i].lower() == exp.lower())
syn_correct_glove50 = sum(1 for i, (_, _, _, exp) in enumerate(SYNTACTIC_ANALOGIES)
                          if syn_results_glove50[i].lower() == exp.lower())

syn_acc_glove300 = (syn_correct_glove300 / len(SYNTACTIC_ANALOGIES)) * 100
syn_acc_glove100 = (syn_correct_glove100 / len(SYNTACTIC_ANALOGIES)) * 100
syn_acc_glove50 = (syn_correct_glove50 / len(SYNTACTIC_ANALOGIES)) * 100


print("ACCURACY SUMMARY")
print(f"{'Analogy Type':<20} {'GloVe 300':<20} {'GloVe 100':<20} {'GloVe 50':<20}")
print("-"*80)
print(f"{'Semantic':<20} {sem_acc_glove300:<19.2f}% {sem_acc_glove100:<19.2f}% {sem_acc_glove50:<19.2f}%")
print(f"{'Syntactic':<20} {syn_acc_glove300:<19.2f}% {syn_acc_glove100:<19.2f}% {syn_acc_glove50:<19.2f}%")

ACCURACY SUMMARY
Analogy Type         GloVe 300            GloVe 100            GloVe 50            
--------------------------------------------------------------------------------
Semantic             50.00              % 40.00              % 30.00              %
Syntactic            60.00              % 50.00              % 40.00              %
